# Multi-Hop Subliminal Learning — Analysis
Starting point for exploration. Uses `scripts.analysis` + the raw metrics.

In [ ]:
import os, sys
from pathlib import Path

# Find the repo root (dir containing scripts/) and work from there.
ROOT = Path.cwd()
while not (ROOT / 'scripts').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import matplotlib.pyplot as plt
from scripts import analysis, paths, utils

FAMILY = 'qwen2.5-7b'
SEED = 0

## First-hop validation

In [ ]:
def trait_rate(root, hop, seed=SEED, family=FAMILY):
    p = paths.hop_dir(root, family, seed, hop) / 'metrics.json'
    return utils.read_json(p)['trait_rate'] if p.exists() else None

def trait_name(root, hop=0, seed=SEED, family=FAMILY):
    p = paths.hop_dir(root, family, seed, hop) / 'metadata.json'
    return utils.read_json(p).get('trait', 'trait') if p.exists() else 'trait'

bars = [('base baseline', trait_rate('data/validation-base', 0)),
        ('teacher (hop 0)', trait_rate('data/validation', 0)),
        ('student (hop 1)', trait_rate('data/validation', 1))]
bars = [(n, tr) for n, tr in bars if tr]
trait = trait_name('data/validation', 0)

fig, ax = plt.subplots(figsize=(5, 4))
labels = [n for n, _ in bars]
means = [tr['mean'] for _, tr in bars]
lo = [tr['mean'] - tr['ci_low'] for _, tr in bars]
hi = [tr['ci_high'] - tr['mean'] for _, tr in bars]
ax.bar(labels, means, yerr=[lo, hi], capsize=5,
       color=['#b0b0b0', '#4c72b0', '#dd8452'][:len(bars)])
ax.set_ylabel(f'{trait} rate'); ax.set_ylim(0, 1)
ax.set_title(f'First-hop validation ({FAMILY}, seed {SEED})')
plt.show()

## 3-hop validation
Run §5 (`run_chain --n-hops 3` on Vast) and download `data/validation-3hop`. The next cell builds the metrics locally; the rest plots them.

In [ ]:
import subprocess, sys
ROOT3 = 'data/validation-3hop'
# Build metrics.json + summary.parquet from the downloaded raw artifacts
# (run_analysis auto-detects the hop count). Skips if nothing is downloaded.
if (paths.base_reference_dir(ROOT3, FAMILY) / 'base_sequences.jsonl').exists():
    subprocess.run([sys.executable, '-m', 'scripts.run_analysis',
                    '--root', ROOT3, '--family', FAMILY, '--seed', str(SEED)], check=True)
else:
    print(f'No 3-hop artifacts under {ROOT3} — run §5 and pull them first.')

In [ ]:
have3 = (paths.seed_dir(ROOT3, FAMILY, SEED) / 'summary.parquet').exists()
df3 = analysis.load_all_summaries(ROOT3, FAMILY) if have3 else None
print('3-hop summary loaded.' if have3 else 'No 3-hop results — run the cell above.')
df3

In [ ]:
if have3:
    fig, axes = plt.subplots(2, 2, figsize=(11, 8)); a = axes.ravel()
    a[0].plot(df3['hop'], df3['trait_rate'], marker='o')
    a[0].set_title('trait rate'); a[0].set_xlabel('hop')
    a[1].plot(df3['hop'], df3['eas_last'], marker='o', color='C2')
    a[1].set_title('EAS (cos vs teacher direction)'); a[1].set_xlabel('hop')
    for col, mk in [('entangled_data', 'o'), ('entangled_unembedding', 's'),
                    ('entangled_logit', '^')]:
        a[2].plot(df3['hop'], df3[col], marker=mk, label=col.replace('entangled_', ''))
    a[2].set_title('entangled-token freq'); a[2].set_xlabel('hop'); a[2].legend()
    a[3].plot(df3['hop'], df3['divergence_freq_A'], marker='o', label='A: carrier freq')
    a[3].plot(df3['hop'], df3['divergence_rate_B'], marker='s', label='B: divergence rate')
    a[3].set_title('divergence tokens'); a[3].set_xlabel('hop'); a[3].legend()
    fig.tight_layout(); plt.show()

### Correlations & loss

In [ ]:
if have3:
    cols = ['eas_last', 'entangled_data', 'entangled_unembedding',
            'entangled_logit', 'divergence_freq_A', 'divergence_rate_B']
    display(analysis.correlation_table(df3, cols))  # vs trait_rate; only ~4 points here

In [ ]:
if have3:
    analysis.plot_loss_curves(ROOT3, FAMILY, SEED); plt.show()

## 5-hop trait trend (2 epochs)
Trait-only preview (§5.3). Reads per-hop `metrics.json` (trait rate) — no local analysis needed. `trait_score` ran on the instance.

In [ ]:
ROOT5 = 'data/validation-5hop-2epochs'
hops, rates, lo, hi = [], [], [], []
for hop in range(6):
    p = paths.hop_dir(ROOT5, FAMILY, SEED, hop) / 'metrics.json'
    tr = utils.read_json(p).get('trait_rate') if p.exists() else None
    if tr:
        hops.append(hop); rates.append(tr['mean'])
        lo.append(tr['mean'] - tr['ci_low']); hi.append(tr['ci_high'] - tr['mean'])
if hops:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.errorbar(hops, rates, yerr=[lo, hi], marker='o', capsize=4)
    ax.set_xlabel('hop'); ax.set_ylabel('trait rate'); ax.set_ylim(0, 1)
    ax.set_title('5-hop trait trend (2 epochs)'); plt.show()
else:
    print(f'No 5-hop/2-epoch results under {ROOT5} - run §5.3 and pull them.')

## 2x2 factorial: trait (cat/owl) x epochs (2/6)
3 seeds, 5 hops each. `trait_rate_valid` counts synonyms and takes the rate over answers that name an animal at all - students that answer 'Qwen' express no animal preference and would otherwise be scored as a 0.

In [ ]:
import numpy as np
import pandas as pd
dff = analysis.load_factorial('data', FAMILY)
COLORS = {'cat-ep2': '#4c72b0', 'cat-ep6': '#1f3d6b',
          'owl-ep2': '#dd8452', 'owl-ep6': '#8c4a1f'}
print(f'{len(dff)} rows, cells: {sorted(dff.cell.unique()) if len(dff) else "none"}')
dff.head()

### Fig 1 - headline: does the trait survive the chain?

In [ ]:
def _band(ax, d, col, label, color):
    g = d.groupby('hop')[col].agg(['mean', 'min', 'max'])
    ax.plot(g.index, g['mean'], marker='o', label=label, color=color)
    ax.fill_between(g.index, g['min'], g['max'], alpha=0.15, color=color)

if len(dff):
    fig, ax = plt.subplots(figsize=(7, 5))
    for cell in analysis.CELLS:
        d = dff[dff.cell == cell]
        if len(d):
            _band(ax, d, 'trait_rate_valid', cell, COLORS[cell])
    ax.set_xlabel('hop (0 = teacher)'); ax.set_ylabel('trait rate (valid answers)')
    ax.set_ylim(0, 1.05); ax.legend()
    ax.set_title('Trait survival across 5 hops (band = seed min-max)')
    plt.show()

### Fig 2 - why the scoring was corrected
Literal `\bcats?\b` misses 'feline'/'kitten'. The teacher says 'feline' 16% of the time; students collapse onto the literal token - which makes students look like they *exceed* their teacher.

In [ ]:
d = dff[dff.trait == 'cat']
if len(d):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for ax, ep in zip(axes, [2, 6]):
        sub = d[d.epochs == ep]
        for col, lab in [('trait_rate', 'literal (as-run)'),
                         ('trait_rate_syn', '+synonyms'),
                         ('trait_rate_valid', '+synonyms, valid only')]:
            g = sub.groupby('hop')[col].mean()
            ax.plot(g.index, g.values, marker='o', label=lab)
        ax.set_title(f'cat, {ep} epochs'); ax.set_xlabel('hop'); ax.set_ylim(0, 1.05)
    axes[0].set_ylabel('cat rate'); axes[0].legend()
    fig.tight_layout(); plt.show()

### Fig 3 - collateral damage: students stop answering the question

In [ ]:
if len(dff):
    fig, ax = plt.subplots(figsize=(7, 5))
    for cell in analysis.CELLS:
        d = dff[dff.cell == cell]
        if len(d):
            _band(ax, d, 'non_answer_rate', cell, COLORS[cell])
    ax.set_xlabel('hop'); ax.set_ylabel("rate of 'Qwen' non-answers")
    ax.set_ylim(0, 1.05); ax.legend()
    ax.set_title('Fraction of answers naming the model instead of an animal')
    plt.show()

### Fig 4 - per-seed, not just the mean
cat-ep2's seed spread is wide; the mean hides it.

In [ ]:
if len(dff):
    fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
    for ax, cell in zip(axes, analysis.CELLS):
        d = dff[dff.cell == cell]
        for seed, g in d.groupby('seed'):
            g = g.sort_values('hop')
            ax.plot(g['hop'], g['trait_rate_valid'], marker='o', label=f'seed {seed}')
        ax.set_title(cell); ax.set_xlabel('hop'); ax.set_ylim(0, 1.05)
    axes[0].set_ylabel('trait rate (valid)'); axes[0].legend(fontsize=8)
    fig.tight_layout(); plt.show()

### Fig 5 - mechanistic metrics
Do EAS / entangled / divergence track the trait, or just drift from base?

In [ ]:
METRICS = [('eas_last', 'EAS (cos vs teacher direction)'),
           ('entangled_data', 'entangled tokens (data method)'),
           ('divergence_rate_B', 'divergence rate (B)'),
           ('divergence_freq_A', 'carrier-token freq (A)')]
if len(dff):
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax, (col, title) in zip(axes.ravel(), METRICS):
        for cell in analysis.CELLS:
            d = dff[dff.cell == cell]
            if len(d):
                g = d.groupby('hop')[col].mean()
                ax.plot(g.index, g.values, marker='o', label=cell, color=COLORS[cell])
        ax.set_title(title); ax.set_xlabel('hop')
    axes[0][0].legend(fontsize=8)
    fig.tight_layout(); plt.show()

### Fig 6 - EAS by layer
summary.parquet carries only the headline layer; the profile comes from metrics.json. The headline-layer choice materially changes the number.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
for ax, cell in zip(axes, analysis.CELLS):
    for hop in range(0, 6):
        eas = analysis.load_eas_per_layer(f'data/{cell}', FAMILY, SEED, hop)
        if eas:
            ax.plot(range(len(eas)), eas, label=f'hop {hop}')
    ax.axvline(11, ls='--', c='grey', lw=1)
    ax.set_title(cell); ax.set_xlabel('layer index'); ax.set_ylim(0, 1.05)
axes[0].set_ylabel('EAS'); axes[0].legend(fontsize=7)
fig.suptitle(f'EAS per layer (seed {SEED}); dashed = headline layer 11')
fig.tight_layout(); plt.show()

### Fig 7 - what the students actually answer

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
for ax, cell in zip(axes, analysis.CELLS):
    trait = cell.split('-')[0]
    comp = [analysis.answer_composition(f'data/{cell}', FAMILY, SEED, h, trait)
            for h in range(0, 6)]
    hops = list(range(0, 6)); bottom = np.zeros(len(hops))
    for key, color in [('trait', '#4c72b0'), ('other_animal', '#b0b0b0'),
                       ('non_answer', '#c44e52')]:
        vals = np.array([c[key] for c in comp])
        ax.bar(hops, vals, bottom=bottom, label=key, color=color)
        bottom += vals
    ax.set_title(cell); ax.set_xlabel('hop'); ax.set_ylim(0, 1)
axes[0].set_ylabel('fraction of answers'); axes[0].legend(fontsize=8)
fig.suptitle(f'Answer composition (seed {SEED})')
fig.tight_layout(); plt.show()

### Fig 8 - training loss

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharey=True)
for ax, cell in zip(axes, analysis.CELLS):
    analysis.plot_loss_curves(f'data/{cell}', FAMILY, SEED, ax=ax)
    ax.set_title(cell); ax.legend(fontsize=7)
fig.tight_layout(); plt.show()

### Correlations: within-cell vs pooled
Pooling across cells is invalid here. Owl's pooled r is driven entirely by the teacher-vs-students contrast (owl students are all ~0), and Spearman collapsing relative to Pearson gives it away. Entangled and divergence tokens are trait-specific, so they are not comparable across traits at all.

In [ ]:
COLS = ['eas_last', 'entangled_data', 'entangled_unembedding',
        'entangled_logit', 'divergence_freq_A', 'divergence_rate_B']
if len(dff):
    out = []
    for cell in analysis.CELLS:
        d = dff[dff.cell == cell]
        if len(d) >= 3:
            t = analysis.correlation_table(d, COLS, y_col='trait_rate_valid')
            out.append(t.assign(scope=cell))
    pooled = analysis.correlation_table(dff, COLS, y_col='trait_rate_valid')
    out.append(pooled.assign(scope='POOLED (invalid - shown to expose the confound)'))
    display(pd.concat(out, ignore_index=True)[
        ['scope', 'metric', 'n', 'pearson_r', 'pearson_p', 'spearman_r', 'spearman_p']])